# 🎯 Predicción del Comportamiento del Sistema Péndulo-Carrito
## Estudio Comparativo de Modelos de Machine Learning

---

**Sistema:** Péndulo invertido sobre carrito (4 variables de estado)

**Variables de estado:**
- `Z1` → Ángulo del péndulo (θ)
- `Z2` → Posición del carrito (x)
- `Z3` → Velocidad angular del péndulo (θ̇)
- `Z4` → Velocidad del carrito (ẋ)
- `W`  → Vector de regresores NARX (15 dimensiones)

**Estructura de archivos (carpeta `data/`):**
```
data/
  Angulopend_exp1.mat   → Z1
  Poscar_exp1.mat       → Z2
  Velpend_exp1.mat      → Z3
  Velcar_exp1.mat       → Z4
  W_exp1.mat            → W (15 regresores NARX)
```

---

### 📋 Modelos a comparar
| # | Modelo | Categoría |
|---|--------|-----------|
| 0 | **ZOH** (Zero-Order Hold) | Baseline |
| 1 | **Regresión Lineal** | Lineal |
| 2 | **Ridge / Lasso** | Lineal regularizado |
| 3 | **SVM** (SVR) | Kernel |
| 4 | **Random Forest** | Ensemble |
| 5 | **MLP** (Red neuronal densa) | Deep Learning |
| 6 | **RNN / LSTM** | Recurrente |

> **Nota:** ZOH es la referencia mínima de comparación — cualquier modelo que no supere al ZOH no aporta valor predictivo.


---
## 0. Imports y Configuración Global

In [ ]:
# ============================================================
#  IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Carga de .mat
import scipy.io as sio

# Sklearn
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

# PyTorch para MLP y RNN/LSTM
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Estilo
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800', '#00BCD4', '#E91E63']
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

# Reproducibilidad
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Directorio de datos
DATA_DIR = Path('data')

print('✅ Imports completados correctamente')
print(f'   PyTorch version: {torch.__version__}')
print(f'   CUDA disponible: {torch.cuda.is_available()}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'   Dispositivo: {DEVICE}')

---
## 1. Carga y Exploración de Datos
### 1.1 Lectura de archivos .mat

In [ ]:
# ============================================================
#  FUNCIÓN GENÉRICA DE CARGA .MAT
# ============================================================
def load_mat_variable(filepath):
    """
    Carga un archivo .mat y extrae el primer array numérico encontrado.
    Aplana a 1D si es necesario.
    """
    mat = sio.loadmat(filepath)
    # Filtrar variables del sistema de MATLAB (empiezan con '__')
    keys = [k for k in mat.keys() if not k.startswith('__')]
    print(f"  Archivo: {Path(filepath).name:30s} | Variables: {keys}")
    # Coger la primera variable útil
    data = mat[keys[0]]
    return data


# ============================================================
#  CARGA DEL EXPERIMENTO 1 (SIMULACIÓN)
# ============================================================
print("📂 Cargando datos del experimento 1 (simulación)...\n")

# Ajusta los nombres de variable interna si difieren de los defaults
Z1_raw = load_mat_variable(DATA_DIR / 'Angulopend_exp1.mat')   # Ángulo péndulo
Z2_raw = load_mat_variable(DATA_DIR / 'Poscar_exp1.mat')       # Posición carrito
Z3_raw = load_mat_variable(DATA_DIR / 'Velpend_exp1.mat')      # Velocidad angular
Z4_raw = load_mat_variable(DATA_DIR / 'Velcar_exp1.mat')       # Velocidad carrito
W_raw  = load_mat_variable(DATA_DIR / 'W_exp1.mat')            # Regresores NARX (15D)

# Aplanar a vectores 1D
Z1 = Z1_raw.flatten()
Z2 = Z2_raw.flatten()
Z3 = Z3_raw.flatten()
Z4 = Z4_raw.flatten()

# W debe tener forma (N, 15)
if W_raw.ndim == 1:
    W_raw = W_raw.reshape(-1, 1)
if W_raw.shape[0] < W_raw.shape[1]:
    W_raw = W_raw.T  # Asegurar (N, 15)
W = W_raw

N = len(Z1)
print(f"\n📊 Dimensiones cargadas:")
print(f"   Z1 (ángulo péndulo):     {Z1.shape}")
print(f"   Z2 (posición carrito):   {Z2.shape}")
print(f"   Z3 (vel. angular):       {Z3.shape}")
print(f"   Z4 (vel. carrito):       {Z4.shape}")
print(f"   W  (regresores NARX):    {W.shape}")
print(f"\n   N total de muestras:     {N}")

In [ ]:
# ============================================================
#  CONSTRUCCIÓN DEL DATAFRAME PRINCIPAL
# ============================================================
# Nombres de columnas para W
n_w = W.shape[1]
w_cols = [f'w{i+1}' for i in range(n_w)]

# DataFrame con señales de estado
df_states = pd.DataFrame({
    'Z1_angle':    Z1,
    'Z2_pos':      Z2,
    'Z3_angvel':   Z3,
    'Z4_vel':      Z4,
})

# DataFrame con regresores W
df_W = pd.DataFrame(W, columns=w_cols)

# DataFrame completo
df = pd.concat([df_states, df_W], axis=1)

print(f'✅ DataFrame construido: {df.shape[0]} filas × {df.shape[1]} columnas')
df.head()

### 1.2 Estadísticas descriptivas

In [ ]:
print("📈 Estadísticas descriptivas de las variables de estado:")
df[['Z1_angle','Z2_pos','Z3_angvel','Z4_vel']].describe().round(4)

In [ ]:
print("📈 Estadísticas descriptivas de los regresores W:")
df_W.describe().round(4)

### 1.3 Visualización de las series temporales

In [ ]:
t = np.arange(N)  # índice temporal (en muestras)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
titles = [
    ('Z1 — Ángulo del péndulo', 'θ (rad)', COLORS[0]),
    ('Z2 — Posición del carrito', 'x (m)', COLORS[1]),
    ('Z3 — Velocidad angular', 'θ̇ (rad/s)', COLORS[2]),
    ('Z4 — Velocidad del carrito', 'ẋ (m/s)', COLORS[3]),
]
signals = [Z1, Z2, Z3, Z4]

for ax, (title, ylabel, color), sig in zip(axes, titles, signals):
    ax.plot(t, sig, color=color, linewidth=0.8, alpha=0.85)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')

axes[-1].set_xlabel('Muestras (k)', fontsize=11)
fig.suptitle('Serie Temporal — Variables de Estado (Experimento 1)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 1.4 Análisis de correlación entre regresores W

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Heatmap de correlación entre regresores W
corr_W = df_W.corr()
mask = np.triu(np.ones_like(corr_W, dtype=bool))
sns.heatmap(corr_W, mask=mask, ax=axes[0], cmap='coolwarm', center=0,
            annot=True if n_w <= 15 else False, fmt='.2f', linewidths=0.3,
            vmin=-1, vmax=1)
axes[0].set_title('Correlación entre regresores W', fontweight='bold')

# Correlación de W con cada variable de estado
corr_WZ = pd.DataFrame({
    'Z1_angle':  [df_W[c].corr(df_states['Z1_angle']) for c in w_cols],
    'Z2_pos':    [df_W[c].corr(df_states['Z2_pos']) for c in w_cols],
    'Z3_angvel': [df_W[c].corr(df_states['Z3_angvel']) for c in w_cols],
    'Z4_vel':    [df_W[c].corr(df_states['Z4_vel']) for c in w_cols],
}, index=w_cols)

sns.heatmap(corr_WZ, ax=axes[1], cmap='coolwarm', center=0,
            annot=True, fmt='.2f', linewidths=0.3, vmin=-1, vmax=1)
axes[1].set_title('Correlación W → Variables de Estado', fontweight='bold')

plt.suptitle('Análisis de Correlación', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 2. Preprocesamiento
### 2.1 Detección y eliminación de columnas redundantes (física + PCA)

In [ ]:
# ============================================================
#  CRITERIO 1: Eliminación por alta correlación (> umbral)
#  Columnas de W con |r| > 0.98 entre sí → redundantes
# ============================================================
CORR_THRESHOLD = 0.98

corr_matrix = df_W.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
cols_to_drop_corr = [col for col in upper.columns if any(upper[col] > CORR_THRESHOLD)]

print(f'🔍 Columnas de W eliminadas por alta correlación (|r| > {CORR_THRESHOLD}): {cols_to_drop_corr}')

df_W_filtered = df_W.drop(columns=cols_to_drop_corr)
print(f'   W: {df_W.shape[1]} → {df_W_filtered.shape[1]} columnas')

In [ ]:
# ============================================================
#  CRITERIO 2: PCA sobre W para analizar varianza explicada
# ============================================================
scaler_pca = StandardScaler()
W_scaled_pca = scaler_pca.fit_transform(df_W_filtered)

pca = PCA()
pca.fit(W_scaled_pca)
explained = np.cumsum(pca.explained_variance_ratio_)

n_components_95 = np.argmax(explained >= 0.95) + 1
n_components_99 = np.argmax(explained >= 0.99) + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(pca.explained_variance_ratio_)+1),
            pca.explained_variance_ratio_ * 100, color=COLORS[0], alpha=0.75)
axes[0].set_xlabel('Componente Principal')
axes[0].set_ylabel('Varianza explicada (%)')
axes[0].set_title('Varianza por Componente (PCA sobre W)')

axes[1].plot(range(1, len(explained)+1), explained * 100, 'o-', color=COLORS[1], linewidth=2)
axes[1].axhline(95, color='gray', linestyle='--', label='95%')
axes[1].axhline(99, color='black', linestyle='--', label='99%')
axes[1].axvline(n_components_95, color=COLORS[2], linestyle=':', alpha=0.8)
axes[1].axvline(n_components_99, color=COLORS[3], linestyle=':', alpha=0.8)
axes[1].set_xlabel('Número de Componentes')
axes[1].set_ylabel('Varianza acumulada (%)')
axes[1].set_title('Varianza Acumulada (PCA sobre W)')
axes[1].legend()
axes[1].set_ylim([0, 101])

plt.tight_layout()
plt.show()

print(f'📌 Componentes para explicar 95% de varianza: {n_components_95}')
print(f'📌 Componentes para explicar 99% de varianza: {n_components_99}')
print(f'   → Reducción potencial: {df_W_filtered.shape[1]} → {n_components_95} dimensiones (95%)')

In [ ]:
# ============================================================
#  CRITERIO 3: Justificación física de los regresores NARX
# ============================================================
# En un sistema NARX de orden p con 4 salidas y 1 entrada (u),
# W contiene: [Z1(k-1)..Z1(k-p), Z2(k-1)..Z2(k-p),
#              Z3(k-1)..Z3(k-p), Z4(k-1)..Z4(k-p), u(k-1)..u(k-pu)]
#
# Con 15 dimensiones y 4 salidas: p=3, pu=3 ó p=3, pu=3
# → Todos los regresores son físicamente relevantes
#   (retardos del estado + entrada)
#
# Sin embargo, Z3 ≈ dZ1/dt y Z4 ≈ dZ2/dt → pueden estar correlacionadas
# con diferencias de Z1 y Z2 (ya incluidas en W), pero NO son redundantes
# porque W incluye valores pasados, no las derivadas calculadas.

print("📐 Justificación física de los 15 regresores NARX:")
print("   Si W = [Z1(k-1), Z1(k-2), Z1(k-3),")
print("           Z2(k-1), Z2(k-2), Z2(k-3),")
print("           Z3(k-1), Z3(k-2), Z3(k-3),")
print("           Z4(k-1), Z4(k-2), Z4(k-3),")
print("           u(k-1), u(k-2), u(k-3)]")
print()
print("   → 12 retardos de estado + 3 retardos de entrada = 15 ✅")
print()
print("   Relación física Z3 ≈ ΔZ1/Ts y Z4 ≈ ΔZ2/Ts")
print("   Verificamos la correlación entre Z3(k-1) y [Z1(k-1)-Z1(k-2)]/Ts:")

if n_w >= 7:
    # Intentar verificar: w4 = Z1(k-1), w5=Z1(k-2), w7=Z3(k-1) (asumiendo orden arriba)
    diff_Z1 = df_W.iloc[:, 0].values - df_W.iloc[:, 1].values  # Z1(k-1) - Z1(k-2)
    corr_check = np.corrcoef(diff_Z1, df_W.iloc[:, 6].values)[0,1]
    print(f"   Correlación Pearson [Z1(k-1)-Z1(k-2)] vs w7: {corr_check:.4f}")
    if abs(corr_check) > 0.9:
        print("   ⚠️  Alta correlación detectada → considerar eliminar columna derivada")
    else:
        print("   ✅ Correlación baja → ambos regresores son informativos independientemente")

### 2.2 Detección de outliers y limpieza

In [ ]:
# ============================================================
#  DETECCIÓN DE OUTLIERS (Z-score sobre variables de estado)
# ============================================================
from scipy import stats

state_cols = ['Z1_angle', 'Z2_pos', 'Z3_angvel', 'Z4_vel']
z_scores = np.abs(stats.zscore(df_states[state_cols]))
Z_THRESH = 4.0  # Umbral conservador para sistemas dinámicos

outlier_mask = (z_scores > Z_THRESH).any(axis=1)
n_outliers = outlier_mask.sum()

print(f'🔎 Outliers detectados (|Z-score| > {Z_THRESH}): {n_outliers} / {N} muestras ({100*n_outliers/N:.2f}%)')

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, col, color in zip(axes.flat, state_cols, COLORS):
    vals = df_states[col].values
    ax.plot(t, vals, color=color, linewidth=0.7, alpha=0.7, label='Normal')
    if n_outliers > 0:
        ax.scatter(t[outlier_mask], vals[outlier_mask],
                   color='red', s=20, zorder=5, label='Outlier')
    ax.set_title(col, fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_xlabel('Muestras')

plt.suptitle('Detección de Outliers (Z-score)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Máscara válida
valid_mask = ~outlier_mask
print(f'✅ Muestras válidas tras limpieza: {valid_mask.sum()}')

### 2.3 Preparación de X (features) e Y (targets) para ML

In [ ]:
# ============================================================
#  DEFINICIÓN DE X e Y
#
#  X = W(k)        → vector de regresores NARX en instante k
#  Y = Z(k+1)      → estado siguiente a predecir
#
#  Esto modela: ẑ(k+1) = f(W(k))
# ============================================================

# Usar solo muestras válidas
W_clean  = W[valid_mask]
Z1_clean = Z1[valid_mask]
Z2_clean = Z2[valid_mask]
Z3_clean = Z3[valid_mask]
Z4_clean = Z4[valid_mask]

# X = W(0..N-2), Y = estado(1..N-1) → predicción un paso adelante
X = W_clean[:-1]         # (N-1, 15)
Y = np.column_stack([
    Z1_clean[1:],        # θ(k+1)
    Z2_clean[1:],        # x(k+1)
    Z3_clean[1:],        # θ̇(k+1)
    Z4_clean[1:],        # ẋ(k+1)
])                       # (N-1, 4)

print(f'📐 Dimensiones del problema ML:')
print(f'   X (features): {X.shape}  → regresores NARX')
print(f'   Y (targets):  {Y.shape}  → [θ, x, θ̇, ẋ] en k+1')

# ============================================================
#  SPLIT TEMPORAL (respeta causalidad — NO shuffle)
# ============================================================
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15  # Los últimos 15% son test

n_total = X.shape[0]
n_train = int(n_total * TRAIN_RATIO)
n_val   = int(n_total * VAL_RATIO)

X_train = X[:n_train]
Y_train = Y[:n_train]

X_val   = X[n_train:n_train+n_val]
Y_val   = Y[n_train:n_train+n_val]

X_test  = X[n_train+n_val:]
Y_test  = Y[n_train+n_val:]

print(f'\n✂️  Split temporal (sin shuffle — respeta causalidad):')
print(f'   Train:      {X_train.shape[0]:5d} muestras ({TRAIN_RATIO*100:.0f}%)')
print(f'   Validación: {X_val.shape[0]:5d} muestras ({VAL_RATIO*100:.0f}%)')
print(f'   Test:       {X_test.shape[0]:5d} muestras ({(1-TRAIN_RATIO-VAL_RATIO)*100:.0f}%)')

# ============================================================
#  ESCALADO (fit solo en train)
# ============================================================
scaler_X = StandardScaler()
scaler_Y = StandardScaler()

X_train_s = scaler_X.fit_transform(X_train)
X_val_s   = scaler_X.transform(X_val)
X_test_s  = scaler_X.transform(X_test)

Y_train_s = scaler_Y.fit_transform(Y_train)
Y_val_s   = scaler_Y.transform(Y_val)
Y_test_s  = scaler_Y.transform(Y_test)

print('\n✅ Escalado completado (StandardScaler fit en train únicamente)')

---
## 3. Definición de Métricas

In [ ]:
# ============================================================
#  FUNCIÓN DE MÉTRICAS COMPLETA
# ============================================================
TARGET_NAMES = ['θ (Z1)', 'x (Z2)', 'θ̇ (Z3)', 'ẋ (Z4)']

def compute_metrics(Y_true, Y_pred, model_name='Modelo'):
    """
    Calcula métricas por variable y globales.
    
    Métricas:
    - RMSE: Error cuadrático medio — penaliza errores grandes
    - MAE:  Error absoluto medio — interpretable en unidades originales
    - R²:   Coef. determinación — varianza explicada (1=perfecto)
    - NRMSE: RMSE normalizado por rango — comparable entre variables
    """
    results = {}
    rows = []
    
    for i, name in enumerate(TARGET_NAMES):
        y_t = Y_true[:, i]
        y_p = Y_pred[:, i]
        rmse  = np.sqrt(mean_squared_error(y_t, y_p))
        mae   = mean_absolute_error(y_t, y_p)
        r2    = r2_score(y_t, y_p)
        rng   = y_t.max() - y_t.min()
        nrmse = rmse / rng if rng != 0 else np.nan
        rows.append({'Variable': name, 'RMSE': rmse, 'MAE': mae,
                     'R²': r2, 'NRMSE': nrmse})
    
    df_metrics = pd.DataFrame(rows).set_index('Variable')
    # Métricas globales (promedio)
    global_row = df_metrics.mean().to_frame().T
    global_row.index = ['GLOBAL (media)']
    df_metrics = pd.concat([df_metrics, global_row])
    df_metrics['Modelo'] = model_name
    return df_metrics


# Diccionario global para almacenar resultados de todos los modelos
ALL_METRICS  = {}   # dict: model_name → df_metrics
ALL_PREDS    = {}   # dict: model_name → Y_pred (en escala original)

print('✅ Función de métricas definida')
print('   Métricas: RMSE | MAE | R² | NRMSE')

---
## 4. Modelo 0 — Baseline: ZOH (Zero-Order Hold)

> **ZOH:** La predicción del estado en $k+1$ es igual al estado en $k$.
> $$\hat{z}(k+1) = z(k)$$
> Es el predictor más simple posible. **Cualquier modelo ML debe superarlo** para ser útil.


In [ ]:
# ============================================================
#  ZOH: ẑ(k+1) = Z(k)  →  usar el estado ACTUAL como predicción
#  Necesitamos Z(k) en el conjunto de test
# ============================================================
# Índices de test (respecto al array limpio)
idx_test_start = n_train + n_val

# Estado actual en test = Z(idx_test_start .. N-2)
Y_zoh = np.column_stack([
    Z1_clean[idx_test_start:-1],
    Z2_clean[idx_test_start:-1],
    Z3_clean[idx_test_start:-1],
    Z4_clean[idx_test_start:-1],
])

# Asegurar misma longitud que Y_test
min_len = min(len(Y_zoh), len(Y_test))
Y_zoh   = Y_zoh[:min_len]
Y_test_eval = Y_test[:min_len]

metrics_zoh = compute_metrics(Y_test_eval, Y_zoh, model_name='ZOH')
ALL_METRICS['ZOH'] = metrics_zoh
ALL_PREDS['ZOH']   = Y_zoh

print("📊 Métricas ZOH (conjunto test):")
metrics_zoh[['RMSE','MAE','R²','NRMSE']].round(5)

In [ ]:
# ============================================================
#  FUNCIÓN DE GRÁFICO PREDICCIÓN vs REAL (reutilizable)
# ============================================================
def plot_prediction_vs_real(Y_true, Y_pred, model_name, n_show=300):
    """
    Grafica predicción vs real para las 4 variables de estado.
    También muestra el error absoluto.
    """
    n = min(n_show, len(Y_true))
    fig = plt.figure(figsize=(16, 10))
    gs  = gridspec.GridSpec(4, 2, hspace=0.45, wspace=0.3)
    
    for i, name in enumerate(TARGET_NAMES):
        ax_main  = fig.add_subplot(gs[i, 0])
        ax_error = fig.add_subplot(gs[i, 1])
        
        ax_main.plot(Y_true[:n, i], color='black', linewidth=1.0,
                     label='Real', alpha=0.85)
        ax_main.plot(Y_pred[:n, i], color=COLORS[i % len(COLORS)],
                     linewidth=1.0, linestyle='--', label=model_name, alpha=0.85)
        ax_main.set_title(f'{name} — Predicción vs Real', fontsize=9, fontweight='bold')
        ax_main.legend(fontsize=8)
        ax_main.set_xlabel('k')
        
        error = np.abs(Y_true[:n, i] - Y_pred[:n, i])
        ax_error.fill_between(range(n), error, color=COLORS[i % len(COLORS)], alpha=0.4)
        ax_error.plot(error, color=COLORS[i % len(COLORS)], linewidth=0.8)
        ax_error.set_title(f'{name} — Error absoluto', fontsize=9)
        ax_error.set_xlabel('k')
        ax_error.set_ylabel('|error|')
    
    fig.suptitle(f'Modelo: {model_name}', fontsize=14, fontweight='bold')
    plt.show()


plot_prediction_vs_real(Y_test_eval, Y_zoh, 'ZOH (baseline)')

---
## 5. Modelo 1 — Regresión Lineal

In [ ]:
# ============================================================
#  REGRESIÓN LINEAL MULTIVARIABLE
#  ẑ(k+1) = A·W(k) + b
# ============================================================
lr_model = LinearRegression()
lr_model.fit(X_train_s, Y_train_s)

Y_pred_lr_s = lr_model.predict(X_test_s[:min_len])
Y_pred_lr   = scaler_Y.inverse_transform(Y_pred_lr_s)

metrics_lr = compute_metrics(Y_test_eval, Y_pred_lr, 'Regresión Lineal')
ALL_METRICS['Linear Regression'] = metrics_lr
ALL_PREDS['Linear Regression']   = Y_pred_lr

print("📊 Métricas Regresión Lineal:")
metrics_lr[['RMSE','MAE','R²','NRMSE']].round(5)

In [ ]:
# Visualización coeficientes
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for i, (ax, name) in enumerate(zip(axes.flat, TARGET_NAMES)):
    coef = lr_model.coef_[i]
    colors_bar = ['#2196F3' if c >= 0 else '#FF5722' for c in coef]
    ax.bar(range(len(coef)), coef, color=colors_bar, alpha=0.8)
    ax.set_title(f'Coeficientes → {name}', fontweight='bold', fontsize=9)
    ax.set_xlabel('Regresor w_i')
    ax.set_ylabel('Coeficiente')
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.set_xticks(range(len(coef)))
    ax.set_xticklabels([f'w{j+1}' for j in range(len(coef))], rotation=45, fontsize=7)

plt.suptitle('Regresión Lineal — Importancia de Regresores', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

plot_prediction_vs_real(Y_test_eval, Y_pred_lr, 'Regresión Lineal')

---
## 6. Modelo 2 — Ridge y Lasso (Regularización)

In [ ]:
# ============================================================
#  RIDGE (L2) y LASSO (L1)
# ============================================================
from sklearn.model_selection import GridSearchCV

alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

# Ridge
ridge_cv = GridSearchCV(
    MultiOutputRegressor(Ridge()),
    param_grid={'estimator__alpha': alphas},
    cv=3, scoring='neg_mean_squared_error', n_jobs=-1
)
ridge_cv.fit(X_train_s, Y_train_s)
best_ridge_alpha = ridge_cv.best_params_['estimator__alpha']
print(f'✅ Ridge — mejor alpha: {best_ridge_alpha}')

Y_pred_ridge_s = ridge_cv.predict(X_test_s[:min_len])
Y_pred_ridge   = scaler_Y.inverse_transform(Y_pred_ridge_s)
metrics_ridge  = compute_metrics(Y_test_eval, Y_pred_ridge, 'Ridge')
ALL_METRICS['Ridge'] = metrics_ridge
ALL_PREDS['Ridge']   = Y_pred_ridge

# Lasso
lasso_cv = GridSearchCV(
    MultiOutputRegressor(Lasso(max_iter=5000)),
    param_grid={'estimator__alpha': alphas},
    cv=3, scoring='neg_mean_squared_error', n_jobs=-1
)
lasso_cv.fit(X_train_s, Y_train_s)
best_lasso_alpha = lasso_cv.best_params_['estimator__alpha']
print(f'✅ Lasso — mejor alpha: {best_lasso_alpha}')

Y_pred_lasso_s = lasso_cv.predict(X_test_s[:min_len])
Y_pred_lasso   = scaler_Y.inverse_transform(Y_pred_lasso_s)
metrics_lasso  = compute_metrics(Y_test_eval, Y_pred_lasso, 'Lasso')
ALL_METRICS['Lasso'] = metrics_lasso
ALL_PREDS['Lasso']   = Y_pred_lasso

print("\n📊 Métricas Ridge:")
print(metrics_ridge[['RMSE','MAE','R²','NRMSE']].round(5))
print("\n📊 Métricas Lasso:")
print(metrics_lasso[['RMSE','MAE','R²','NRMSE']].round(5))

In [ ]:
plot_prediction_vs_real(Y_test_eval, Y_pred_ridge, 'Ridge')
plot_prediction_vs_real(Y_test_eval, Y_pred_lasso, 'Lasso')

---
## 7. Modelo 3 — SVR (Support Vector Regression)

In [ ]:
# ============================================================
#  SVM / SVR — kernel RBF (el más común para sistemas dinámicos)
#  MultiOutput: un SVR por variable de salida
# ============================================================
# NOTA: SVR es costoso computacionalmente — limitamos muestra si N es grande
MAX_SVR_SAMPLES = 5000

if X_train_s.shape[0] > MAX_SVR_SAMPLES:
    idx_svr = np.random.choice(X_train_s.shape[0], MAX_SVR_SAMPLES, replace=False)
    idx_svr.sort()
    X_svr = X_train_s[idx_svr]
    Y_svr = Y_train_s[idx_svr]
    print(f'⚠️  SVR entrenado con {MAX_SVR_SAMPLES} muestras (subconjunto aleatorio)')
else:
    X_svr, Y_svr = X_train_s, Y_train_s

svr_model = MultiOutputRegressor(
    SVR(kernel='rbf', C=10.0, epsilon=0.01, gamma='scale'),
    n_jobs=-1
)
svr_model.fit(X_svr, Y_svr)

Y_pred_svr_s = svr_model.predict(X_test_s[:min_len])
Y_pred_svr   = scaler_Y.inverse_transform(Y_pred_svr_s)

metrics_svr = compute_metrics(Y_test_eval, Y_pred_svr, 'SVR (RBF)')
ALL_METRICS['SVR (RBF)'] = metrics_svr
ALL_PREDS['SVR (RBF)']   = Y_pred_svr

print("📊 Métricas SVR:")
metrics_svr[['RMSE','MAE','R²','NRMSE']].round(5)

In [ ]:
plot_prediction_vs_real(Y_test_eval, Y_pred_svr, 'SVR (RBF)')

---
## 8. Modelo 4 — Random Forest

In [ ]:
# ============================================================
#  RANDOM FOREST
#  Modelo ensemble basado en árboles — captura no linealidades
# ============================================================
rf_model = MultiOutputRegressor(
    RandomForestRegressor(n_estimators=200, max_depth=12,
                          min_samples_leaf=3, random_state=SEED, n_jobs=-1),
    n_jobs=-1
)
rf_model.fit(X_train_s, Y_train_s)

Y_pred_rf_s = rf_model.predict(X_test_s[:min_len])
Y_pred_rf   = scaler_Y.inverse_transform(Y_pred_rf_s)

metrics_rf = compute_metrics(Y_test_eval, Y_pred_rf, 'Random Forest')
ALL_METRICS['Random Forest'] = metrics_rf
ALL_PREDS['Random Forest']   = Y_pred_rf

print("📊 Métricas Random Forest:")
metrics_rf[['RMSE','MAE','R²','NRMSE']].round(5)

In [ ]:
# Importancia de características (feature importance)
importances = np.mean(
    [est.feature_importances_ for est in rf_model.estimators_], axis=0
)

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(range(len(importances)), importances, color=COLORS[4], alpha=0.8)
ax.set_xticks(range(len(importances)))
ax.set_xticklabels([f'w{i+1}' for i in range(len(importances))], rotation=45)
ax.set_xlabel('Regresor')
ax.set_ylabel('Importancia media')
ax.set_title('Random Forest — Importancia de Regresores NARX', fontweight='bold')

# Anotar top 3
top3 = np.argsort(importances)[-3:]
for idx in top3:
    ax.bar(idx, importances[idx], color='#E91E63', alpha=0.9)
    ax.text(idx, importances[idx]+0.001, f'w{idx+1}', ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'🔑 Top 3 regresores más importantes: {[f"w{i+1}" for i in top3[::-1]]}')

plot_prediction_vs_real(Y_test_eval, Y_pred_rf, 'Random Forest')

---
## 9. Modelo 5 — MLP (Red Neuronal Multicapa)

Arquitectura: **W(15) → FC(64) → ReLU → FC(64) → ReLU → FC(4)**

In [ ]:
# ============================================================
#  MLP CON PYTORCH
# ============================================================

class MLP(nn.Module):
    def __init__(self, in_dim=15, hidden_dims=(128, 64, 32), out_dim=4, dropout=0.1):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
            prev = h
        layers.append(nn.Linear(prev, out_dim))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)


def train_pytorch_model(model, X_tr, Y_tr, X_v, Y_v,
                         epochs=150, batch_size=256, lr=1e-3, patience=20):
    """
    Entrena un modelo PyTorch con early stopping basado en val_loss.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
    criterion = nn.MSELoss()
    
    X_tr_t = torch.FloatTensor(X_tr).to(DEVICE)
    Y_tr_t = torch.FloatTensor(Y_tr).to(DEVICE)
    X_v_t  = torch.FloatTensor(X_v).to(DEVICE)
    Y_v_t  = torch.FloatTensor(Y_v).to(DEVICE)
    
    dataset   = TensorDataset(X_tr_t, Y_tr_t)
    loader    = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    best_val_loss = np.inf
    best_state    = None
    patience_cnt  = 0
    train_losses, val_losses = [], []
    
    model.train()
    for epoch in range(epochs):
        model.train()
        ep_loss = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            ep_loss += loss.item() * len(xb)
        ep_loss /= len(X_tr)
        
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_v_t), Y_v_t).item()
        
        scheduler.step(val_loss)
        train_losses.append(ep_loss)
        val_losses.append(val_loss)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                print(f'   Early stopping en epoch {epoch+1}')
                break
        
        if (epoch+1) % 25 == 0:
            print(f'   Epoch {epoch+1:4d} | train_loss: {ep_loss:.6f} | val_loss: {val_loss:.6f}')
    
    model.load_state_dict(best_state)
    return model, train_losses, val_losses


print('🧠 Entrenando MLP...')
mlp = MLP(in_dim=X_train_s.shape[1], hidden_dims=(128, 64, 32), out_dim=4).to(DEVICE)
mlp, mlp_train_loss, mlp_val_loss = train_pytorch_model(
    mlp, X_train_s, Y_train_s, X_val_s, Y_val_s,
    epochs=200, batch_size=256, lr=1e-3, patience=25
)

mlp.eval()
with torch.no_grad():
    Y_pred_mlp_s = mlp(torch.FloatTensor(X_test_s[:min_len]).to(DEVICE)).cpu().numpy()
Y_pred_mlp = scaler_Y.inverse_transform(Y_pred_mlp_s)

metrics_mlp = compute_metrics(Y_test_eval, Y_pred_mlp, 'MLP')
ALL_METRICS['MLP'] = metrics_mlp
ALL_PREDS['MLP']   = Y_pred_mlp

print("\n📊 Métricas MLP:")
print(metrics_mlp[['RMSE','MAE','R²','NRMSE']].round(5))

In [ ]:
# Curva de aprendizaje MLP
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(mlp_train_loss, label='Train loss', color=COLORS[0], linewidth=1.5)
ax.plot(mlp_val_loss,   label='Val loss',   color=COLORS[1], linewidth=1.5, linestyle='--')
ax.set_xlabel('Época')
ax.set_ylabel('MSE loss')
ax.set_title('MLP — Curva de Aprendizaje', fontweight='bold')
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.show()

plot_prediction_vs_real(Y_test_eval, Y_pred_mlp, 'MLP')

---
## 10. Modelo 6 — LSTM (Long Short-Term Memory)

La LSTM procesa **secuencias de longitud `seq_len`** de W para predecir Z(k+1). Captura dependencias temporales largas que el MLP no puede modelar.

In [ ]:
# ============================================================
#  PREPARACIÓN DE DATOS SECUENCIALES PARA LSTM
# ============================================================
SEQ_LEN = 10  # Ventana temporal: W(k-seq+1), ..., W(k) → Z(k+1)

def make_sequences(X_arr, Y_arr, seq_len):
    """
    Crea secuencias temporales para LSTM.
    Input:  X (N, features), Y (N, targets)
    Output: Xs (N-seq, seq, features), Ys (N-seq, targets)
    """
    Xs, Ys = [], []
    for i in range(seq_len, len(X_arr)):
        Xs.append(X_arr[i-seq_len:i])
        Ys.append(Y_arr[i])
    return np.array(Xs), np.array(Ys)


# Concatenar para crear secuencias continuas (respetando split temporal)
X_seq_train, Y_seq_train = make_sequences(X_train_s, Y_train_s, SEQ_LEN)
X_seq_val,   Y_seq_val   = make_sequences(
    np.vstack([X_train_s[-SEQ_LEN:], X_val_s]),
    np.vstack([Y_train_s[-SEQ_LEN:], Y_val_s]),
    SEQ_LEN
)
X_seq_test, Y_seq_test = make_sequences(
    np.vstack([X_val_s[-SEQ_LEN:], X_test_s[:min_len]]),
    np.vstack([Y_val_s[-SEQ_LEN:], Y_test_s[:min_len]]),
    SEQ_LEN
)

print(f'📐 Secuencias LSTM:')
print(f'   Train: {X_seq_train.shape}')
print(f'   Val:   {X_seq_val.shape}')
print(f'   Test:  {X_seq_test.shape}')

In [ ]:
# ============================================================
#  ARQUITECTURA LSTM
# ============================================================
class LSTMModel(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, num_layers=2, out_dim=4, dropout=0.15):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=in_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32, out_dim)
        )
    
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])  # Último estado oculto


def train_seq_model(model, X_tr, Y_tr, X_v, Y_v,
                    epochs=200, batch_size=128, lr=1e-3, patience=25):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
    criterion = nn.MSELoss()
    
    X_tr_t = torch.FloatTensor(X_tr).to(DEVICE)
    Y_tr_t = torch.FloatTensor(Y_tr).to(DEVICE)
    X_v_t  = torch.FloatTensor(X_v).to(DEVICE)
    Y_v_t  = torch.FloatTensor(Y_v).to(DEVICE)
    
    loader = DataLoader(TensorDataset(X_tr_t, Y_tr_t), batch_size=batch_size, shuffle=True)
    best_val, best_state, patience_cnt = np.inf, None, 0
    train_losses, val_losses = [], []
    
    for epoch in range(epochs):
        model.train()
        ep_loss = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping
            optimizer.step()
            ep_loss += loss.item() * len(xb)
        ep_loss /= len(X_tr)
        
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_v_t), Y_v_t).item()
        
        scheduler.step(val_loss)
        train_losses.append(ep_loss)
        val_losses.append(val_loss)
        
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                print(f'   Early stopping en epoch {epoch+1}')
                break
        
        if (epoch+1) % 25 == 0:
            print(f'   Epoch {epoch+1:4d} | train: {ep_loss:.6f} | val: {val_loss:.6f}')
    
    model.load_state_dict(best_state)
    return model, train_losses, val_losses


print('🧠 Entrenando LSTM...')
lstm = LSTMModel(in_dim=X_train_s.shape[1], hidden_dim=64, num_layers=2, out_dim=4).to(DEVICE)
lstm, lstm_train_loss, lstm_val_loss = train_seq_model(
    lstm, X_seq_train, Y_seq_train, X_seq_val, Y_seq_val,
    epochs=200, batch_size=128, lr=1e-3, patience=25
)

lstm.eval()
with torch.no_grad():
    Y_pred_lstm_s = lstm(torch.FloatTensor(X_seq_test).to(DEVICE)).cpu().numpy()
Y_pred_lstm = scaler_Y.inverse_transform(Y_pred_lstm_s)

# Alinear con Y_test_eval
min_len_lstm = min(len(Y_pred_lstm), len(Y_test_eval))
Y_pred_lstm  = Y_pred_lstm[:min_len_lstm]
Y_test_lstm  = Y_test_eval[:min_len_lstm]

metrics_lstm = compute_metrics(Y_test_lstm, Y_pred_lstm, 'LSTM')
ALL_METRICS['LSTM'] = metrics_lstm
ALL_PREDS['LSTM']   = Y_pred_lstm

print("\n📊 Métricas LSTM:")
print(metrics_lstm[['RMSE','MAE','R²','NRMSE']].round(5))

In [ ]:
# Curva de aprendizaje LSTM
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lstm_train_loss, label='Train loss', color=COLORS[5], linewidth=1.5)
ax.plot(lstm_val_loss,   label='Val loss',   color=COLORS[6], linewidth=1.5, linestyle='--')
ax.set_xlabel('Época')
ax.set_ylabel('MSE loss')
ax.set_title('LSTM — Curva de Aprendizaje', fontweight='bold')
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.show()

plot_prediction_vs_real(Y_test_lstm, Y_pred_lstm, 'LSTM')

---
## 11. 🏆 Comparación Global de Modelos vs ZOH

Esta sección es el **núcleo del análisis**. Comparamos todos los modelos frente al ZOH.

In [ ]:
# ============================================================
#  TABLA COMPARATIVA GLOBAL
# ============================================================
rows_global = []
for model_name, df_m in ALL_METRICS.items():
    row = df_m.loc['GLOBAL (media)'].copy()
    row.name = model_name
    rows_global.append(row[['RMSE','MAE','R²','NRMSE']])

df_comparison = pd.DataFrame(rows_global)
df_comparison = df_comparison.sort_values('RMSE')

# Calcular mejora relativa respecto a ZOH
zoh_rmse = df_comparison.loc['ZOH', 'RMSE']
zoh_r2   = df_comparison.loc['ZOH', 'R²']
df_comparison['ΔRMSE vs ZOH (%)'] = (
    (df_comparison['RMSE'] - zoh_rmse) / zoh_rmse * 100
).round(2)
df_comparison['ΔR² vs ZOH'] = (df_comparison['R²'] - zoh_r2).round(4)

print("🏆 Tabla Comparativa Global (ordenada por RMSE):")
print("   ΔRMSE negativo = mejora sobre ZOH ✅")
print("   ΔRMSE positivo = peor que ZOH ❌")
print()
df_comparison.round(5)

In [ ]:
# ============================================================
#  GRÁFICO 1: Barras comparativas RMSE por variable
# ============================================================
model_names  = list(ALL_METRICS.keys())
n_models     = len(model_names)
bar_colors   = COLORS[:n_models]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for idx, (ax, target) in enumerate(zip(axes.flat, TARGET_NAMES)):
    rmse_vals = []
    for mname in model_names:
        dm = ALL_METRICS[mname]
        # Buscar la fila con el target
        if target in dm.index:
            rmse_vals.append(dm.loc[target, 'RMSE'])
        else:
            rmse_vals.append(np.nan)
    
    bars = ax.bar(range(n_models), rmse_vals, color=bar_colors, alpha=0.85, edgecolor='white')
    
    # Línea de referencia ZOH
    zoh_idx = model_names.index('ZOH') if 'ZOH' in model_names else 0
    ax.axhline(rmse_vals[zoh_idx], color='red', linewidth=2, linestyle='--',
               label=f'ZOH = {rmse_vals[zoh_idx]:.4f}')
    
    ax.set_xticks(range(n_models))
    ax.set_xticklabels(model_names, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('RMSE')
    ax.set_title(f'RMSE — {target}', fontweight='bold')
    ax.legend(fontsize=8)
    
    # Anotar valores
    for b, v in zip(bars, rmse_vals):
        if not np.isnan(v):
            ax.text(b.get_x() + b.get_width()/2, v, f'{v:.4f}',
                    ha='center', va='bottom', fontsize=7, rotation=30)

plt.suptitle('Comparación RMSE por Variable de Estado — Todos los Modelos vs ZOH',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  GRÁFICO 2: R² global por modelo (con línea ZOH)
# ============================================================
r2_global = []
for mname in model_names:
    dm = ALL_METRICS[mname]
    r2_global.append(dm.loc['GLOBAL (media)', 'R²'])

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(range(n_models), r2_global, color=bar_colors, alpha=0.85, edgecolor='white')
ax.axhline(r2_global[model_names.index('ZOH')] if 'ZOH' in model_names else 0,
           color='red', linewidth=2.5, linestyle='--', label='ZOH baseline')
ax.axhline(1.0, color='green', linewidth=1, linestyle=':', alpha=0.5, label='R²=1 (perfecto)')
ax.set_xticks(range(n_models))
ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylabel('R² (medio)')
ax.set_title('R² Global por Modelo — Comparación vs ZOH', fontsize=13, fontweight='bold')
ax.legend()
ax.set_ylim([min(r2_global) - 0.05, 1.05])

for b, v in zip(bars, r2_global):
    ax.text(b.get_x() + b.get_width()/2, v + 0.005, f'{v:.4f}',
            ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  GRÁFICO 3: Mejora relativa sobre ZOH (radar chart)
# ============================================================
from matplotlib.patches import FancyArrowPatch

categories = TARGET_NAMES
N_cat = len(categories)
angles = np.linspace(0, 2 * np.pi, N_cat, endpoint=False).tolist()
angles += angles[:1]  # Cerrar el polígono

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

# ZOH como referencia (NRMSE = 1)
zoh_nrmse = [ALL_METRICS['ZOH'].loc[t, 'NRMSE'] if t in ALL_METRICS['ZOH'].index
             else 0 for t in TARGET_NAMES]

for i, mname in enumerate(model_names):
    dm = ALL_METRICS[mname]
    vals = [dm.loc[t, 'NRMSE'] if t in dm.index else 0 for t in TARGET_NAMES]
    # Normalizar respecto a ZOH (ratio < 1 = mejor que ZOH)
    vals_norm = [v / z if z != 0 else 1 for v, z in zip(vals, zoh_nrmse)]
    vals_norm += vals_norm[:1]
    
    ls = '--' if mname == 'ZOH' else '-'
    lw = 2.5 if mname == 'ZOH' else 1.5
    ax.plot(angles, vals_norm, ls, color=bar_colors[i], linewidth=lw, label=mname)
    ax.fill(angles, vals_norm, color=bar_colors[i], alpha=0.07)

ax.set_thetagrids(np.degrees(angles[:-1]), TARGET_NAMES, fontsize=10)
ax.axhline(1.0, color='red', linewidth=1.5, linestyle=':', alpha=0.6)
ax.set_title('NRMSE relativo al ZOH\n(< 1 = mejor que ZOH)', fontsize=12,
             fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  GRÁFICO 4: Predicciones superpuestas — todos los modelos
#  (Un gráfico por variable, primeros 200 muestras de test)
# ============================================================
N_SHOW = 200

fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)

for idx_var, (ax, target_name) in enumerate(zip(axes, TARGET_NAMES)):
    # Real
    ax.plot(Y_test_eval[:N_SHOW, idx_var], color='black', linewidth=1.5,
            label='Real', zorder=10)
    
    # Cada modelo
    for i, (mname, ypred) in enumerate(ALL_PREDS.items()):
        if len(ypred) >= N_SHOW:
            y = ypred[:N_SHOW, idx_var]
        else:
            y = ypred[:, idx_var]
        
        ls = '--' if mname == 'ZOH' else '-'
        lw = 2.0 if mname == 'ZOH' else 1.0
        alpha = 0.9 if mname in ['ZOH', 'LSTM', 'MLP'] else 0.55
        ax.plot(y, color=COLORS[i % len(COLORS)], linewidth=lw,
                linestyle=ls, label=mname, alpha=alpha)
    
    ax.set_title(f'{target_name}', fontweight='bold', fontsize=11)
    ax.set_ylabel('Valor')
    if idx_var == 0:
        ax.legend(loc='upper right', fontsize=8, ncol=len(ALL_PREDS))

axes[-1].set_xlabel('Muestras de test (k)', fontsize=11)
fig.suptitle('Predicciones de Todos los Modelos vs Real (primeras 200 muestras de test)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  GRÁFICO 5: Residuos (histograma por modelo y variable)
# ============================================================
fig, axes = plt.subplots(len(ALL_PREDS), 4, figsize=(16, 3*len(ALL_PREDS)))

for i, (mname, ypred) in enumerate(ALL_PREDS.items()):
    n = min(len(ypred), len(Y_test_eval))
    residuals = Y_test_eval[:n] - ypred[:n]
    for j, target in enumerate(TARGET_NAMES):
        ax = axes[i, j]
        ax.hist(residuals[:, j], bins=40, color=COLORS[i % len(COLORS)],
                alpha=0.75, edgecolor='white')
        ax.axvline(0, color='red', linewidth=1.5)
        ax.set_title(f'{mname}\n{target}', fontsize=8)
        mu = residuals[:, j].mean()
        ax.axvline(mu, color='orange', linewidth=1, linestyle='--')
        ax.set_xlabel('Residuo', fontsize=7)

plt.suptitle('Distribución de Residuos por Modelo', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 12. 📊 Dashboard de Resultados — Tabla Final

In [ ]:
# ============================================================
#  TABLA FINAL DETALLADA POR VARIABLE
# ============================================================
all_rows = []
for mname, dm in ALL_METRICS.items():
    for target in TARGET_NAMES:
        if target in dm.index:
            row = dm.loc[target].copy()
            row['Modelo'] = mname
            row['Variable'] = target
            all_rows.append(row[['Modelo','Variable','RMSE','MAE','R²','NRMSE']])

df_final = pd.DataFrame(all_rows)
df_final = df_final.set_index(['Modelo','Variable']).sort_index()

# Pivot para mejor visualización
df_pivot_rmse = df_final['RMSE'].unstack('Variable')
df_pivot_r2   = df_final['R²'].unstack('Variable')

print("📊 RMSE por modelo y variable:")
print(df_pivot_rmse.round(5).to_string())
print("\n📊 R² por modelo y variable:")
print(df_pivot_r2.round(4).to_string())

In [ ]:
# ============================================================
#  HEATMAP RMSE NORMALIZADO (respecto al ZOH)
# ============================================================
rmse_pivot = df_pivot_rmse.copy()
zoh_row = rmse_pivot.loc['ZOH'] if 'ZOH' in rmse_pivot.index else rmse_pivot.iloc[0]
rmse_norm = rmse_pivot.div(zoh_row)  # Ratio vs ZOH — 1=igual al ZOH, <1=mejor

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(rmse_norm, ax=ax, cmap='RdYlGn_r', center=1.0,
            annot=True, fmt='.3f', linewidths=0.5,
            cbar_kws={'label': 'RMSE / RMSE_ZOH  (< 1 = mejor que ZOH)'})
ax.set_title('RMSE Normalizado respecto al ZOH\n' +
             '🟢 Verde = mejor que ZOH | 🔴 Rojo = peor que ZOH',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Interpretación:")
print("   Ratio < 1.0 → el modelo mejora al ZOH ✅")
print("   Ratio = 1.0 → el modelo equivale al ZOH ⚖️")
print("   Ratio > 1.0 → el modelo es peor que el ZOH ❌")

---
## 13. 🔬 Análisis de Errores Temporales — Degradación con el Horizonte

In [ ]:
# ============================================================
#  RMSE a lo largo del tiempo de test
#  (ventana deslizante de 50 muestras)
# ============================================================
WINDOW = 50

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for idx_var, (ax, target) in enumerate(zip(axes.flat, TARGET_NAMES)):
    for i, (mname, ypred) in enumerate(ALL_PREDS.items()):
        n = min(len(ypred), len(Y_test_eval))
        errors = (Y_test_eval[:n, idx_var] - ypred[:n, idx_var])**2
        # RMSE por ventana
        rmse_t = [
            np.sqrt(errors[j:j+WINDOW].mean())
            for j in range(0, n-WINDOW, WINDOW//2)
        ]
        t_axis = [j + WINDOW//2 for j in range(0, n-WINDOW, WINDOW//2)]
        ls = '--' if mname == 'ZOH' else '-'
        lw = 2 if mname == 'ZOH' else 1.2
        ax.plot(t_axis, rmse_t, ls, color=COLORS[i % len(COLORS)],
                linewidth=lw, label=mname, alpha=0.85)
    
    ax.set_title(f'{target} — RMSE deslizante (ventana={WINDOW})', fontweight='bold')
    ax.set_xlabel('k (muestra de test)')
    ax.set_ylabel('RMSE local')
    if idx_var == 0:
        ax.legend(fontsize=8)

plt.suptitle('Evolución Temporal del Error (RMSE deslizante)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 14. 💾 Extensión: Carga de Múltiples Experimentos

In [ ]:
# ============================================================
#  FUNCIÓN PARA CARGAR MÚLTIPLES EXPERIMENTOS
#  Uso: para datos reales o múltiples repeticiones
# ============================================================
def load_experiment(data_dir, exp_id, prefix=''):
    """
    Carga un experimento completo dado su ID.
    prefix: '' para simulación, 'Real_' para datos reales (ajustar según nombres)
    """
    data_dir = Path(data_dir)
    tag = f'{prefix}exp{exp_id}'
    
    files = {
        'Z1': data_dir / f'Angulopend_{tag}.mat',
        'Z2': data_dir / f'Poscar_{tag}.mat',
        'Z3': data_dir / f'Velpend_{tag}.mat',
        'Z4': data_dir / f'Velcar_{tag}.mat',
        'W':  data_dir / f'W_{tag}.mat',
    }
    
    loaded = {}
    for key, path in files.items():
        if path.exists():
            raw = load_mat_variable(path)
            loaded[key] = raw.flatten() if key != 'W' else (
                raw.T if raw.shape[0] < raw.shape[1] else raw
            )
        else:
            print(f'   ⚠️  No encontrado: {path}')
    return loaded


# Ejemplo de carga de experimento 2 (si existe)
# exp2 = load_experiment(DATA_DIR, 2)

print('✅ Función load_experiment() definida.')
print('   Uso: load_experiment(DATA_DIR, exp_id=2)')
print('   Para datos reales: load_experiment(DATA_DIR, exp_id=1, prefix="Real_")')

---
## 15. 📝 Conclusiones

### Resumen de resultados

In [ ]:
# ============================================================
#  RESUMEN AUTOMÁTICO
# ============================================================
print("="*65)
print("  RESUMEN FINAL — PREDICCIÓN PÉNDULO-CARRITO")
print("="*65)
print()

df_summary = df_comparison[['RMSE','R²','ΔRMSE vs ZOH (%)','ΔR² vs ZOH']].copy()

zoh_rmse_val = df_comparison.loc['ZOH','RMSE'] if 'ZOH' in df_comparison.index else None

for mname in df_comparison.index:
    rmse_val = df_comparison.loc[mname, 'RMSE']
    r2_val   = df_comparison.loc[mname, 'R²']
    delta    = df_comparison.loc[mname, 'ΔRMSE vs ZOH (%)']
    
    if mname == 'ZOH':
        status = '🔴 BASELINE'
    elif delta < -5:
        status = f'✅ MEJORA ({delta:.1f}% vs ZOH)'
    elif delta < 0:
        status = f'✅ Ligera mejora ({delta:.1f}% vs ZOH)'
    else:
        status = f'❌ No supera al ZOH (+{delta:.1f}%)'
    
    print(f'  {mname:20s} | RMSE: {rmse_val:.5f} | R²: {r2_val:.4f} | {status}')

print()
print("-"*65)
best_model = df_comparison[df_comparison.index != 'ZOH']['RMSE'].idxmin()
print(f'  🏆 Mejor modelo: {best_model}')
print(f'     RMSE: {df_comparison.loc[best_model,"RMSE"]:.5f}')
print(f'     R²:   {df_comparison.loc[best_model,"R²"]:.4f}')
print(f'     Mejora sobre ZOH: {df_comparison.loc[best_model,"ΔRMSE vs ZOH (%)"]:.2f}%')
print("="*65)
print()
print("📐 Notas metodológicas:")
print("   • Split temporal 70/15/15 (sin shuffle — respeta causalidad)")
print("   • Escalado StandardScaler fit solo en train")
print("   • ZOH: ẑ(k+1) = z(k) — predictor mínimo de referencia")
print("   • LSTM con ventana temporal de", SEQ_LEN, "muestras")
print("   • Todas las métricas calculadas en el conjunto de TEST")